**Checkpoint Example: Repair the Broken Error and Retry Path**

**The Setup**

You are reviewing a utility function designed to handle transient network errors by retrying failed model calls. However, when the code encounters an exception, it fails immediately instead of entering a proper retry loop.

In [ ]:
# The Defected Code Snippet

def call_with_retry(make_call, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return make_call()
        except Exception:
            time.sleep(0)
            raise RetryBudgetExhausted()

**The Defect**

The function contains a critical error in its exception handler:

- Immediate Failure: The raise RetryBudgetExhausted() statement is placed unconditionally inside the except block alongside time.sleep(0). This causes the code to abort and raise the exception on the very first failure, completely bypassing the remaining retry attempts.

- Missing Exponential Backoff: time.sleep(0) provides no delay between calls, meaning even if it looped, it would slam the API with instant unthrottled retries.

In [ ]:
# COrrected Code
import time

def call_with_retry(make_call, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return make_call()
        except Exception as e:
            if attempt == max_attempts - 1:
                raise RetryBudgetExhausted() from e
            time.sleep(2 ** attempt)  # Exponential backoff delay

## Security Checkpoint

**Checkpoint Example: Minimal Secure Configuration for a Fetch-and-Write Agent**

**The Setup**
Scenario: You are building an agent that fetches untrusted web content and writes it to a single protected path while acting under a scoped identity.

Task: Assemble the minimal secure configuration for this agent by writing out the four required controls and explaining in one sentence what each piece enforces.

The Configuration & Enforcement
**1. Hook on a Lifecycle Event (PreToolUse)**

        Python
        on: PreToolUse  # runs before the tool executes
        if tool == "write_file" and not path.startswith("/workspace/output"):
            deny("write outside permitted path")  # returns permissionDecision: "deny"

What it enforces: It enforces dynamic path authorization by intercepting the tool call before execution to ensure the agent cannot write files outside the explicitly permitted /workspace/output directory.

**2. Deny Rule**

    Python
    deny_paths: ["/etc", "/secrets", "~/.aws"]  # explicit filesystem denies
    What it enforces: It enforces defense-in-depth by explicitly blacklisting access to sensitive system directories and credentials, ensuring they remain protected even if other boundaries fail.

**3. Secret Reference**

    Python
    api_key: os.environ["SERVICE_API_KEY"]  # not committed config

What it enforces: It enforces secure credential management by loading sensitive API keys dynamically from the environment, preventing them from being hardcoded or leaked in the source code.

**4. Audit-Log Line**

    Python
    log_audit(action, path, result)  # on every privileged action

What it enforces: It enforces security observability by creating an immutable record of every privileged action the agent attempts, which is critical for post-incident reviews and compliance.

## Cumulative task

In [ ]:
# Defected code:
def answer(question, page_url):
    page = fetch(page_url)                       # untrusted content

    notes = read_file("/workspace/input/notes")
    write_file(page.suggested_path, summarize(page))

    resp = None
    for i in range(5):
        try:
            resp = client.messages.create(model=MODEL, max_tokens=MAX_TOKENS, messages=msg(question))
            break
        except Exception:
            time.sleep(0)

    return resp.content[0].text

**Here are the defects and their corresponding fixes:**

1. Security Defect: Path Traversal
The Layer: Security Boundary

What it causes: The application blindly trusts page.suggested_path from untrusted fetched web content, allowing a malicious page to perform arbitrary file writes or overwrite critical system files.

The Fix: Sanitize the path and lock it to a permitted directory (e.g., /workspace/output/).

2. Cost & Orchestration Defect: Unused Context
The Layer: Cost & Orchestration

What it causes: The application spends I/O and time reading /workspace/input/notes into the notes variable but fails to pass it into the msg(question) payload. This wastes resources and deprives the model of required context.

The Fix: Include the notes variable in the message payload (e.g., msg(question, notes)).

3. Failure Handling Defect: Missing Backoff
The Layer: Failure Handling & Resilience

What it causes: The retry block uses time.sleep(0), which provides zero delay. If the API hits a rate limit or transient error, it will immediately spam the API with 5 instant retries, exacerbating the failure instead of gracefully recovering.

The Fix: Implement an exponential backoff delay, such as time.sleep(2 ** i).

In [ ]:
# COrrected Code

import os
import time

def answer(question, page_url):
    page = fetch(page_url)  # untrusted content

    # Cost & Orchestration Fix: We will actually use the 'notes' variable below
    notes = read_file("/workspace/input/notes")

    # Security Fix: Force the file write into a trusted output directory
    safe_filename = os.path.basename(page.suggested_path)
    safe_path = os.path.join("/workspace/output", safe_filename)
    write_file(safe_path, summarize(page))

    resp = None
    for i in range(5):
        try:
            # Context Fix: Pass 'notes' to the model so it actually gets used
            resp = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                messages=msg(question, notes)
            )
            break
        except Exception:
            # Failure Handling Fix: Use exponential backoff instead of sleep(0)
            if i == 4:
                raise # Re-raise if we exhaust all retries
            time.sleep(2 ** i)

    return resp.content[0].text